In [1]:
%pip install -r requirements.txt

  Using cached llama_index-0.14.25-py3-none-any.whl.metadata (12 kB)
  Using cached llama_index_embeddings_huggingface-0.8.0-py3-none-any.whl.metadata (459 bytes)
  Using cached sentence_transformers-6.1.0-py3-none-any.whl.metadata (20 kB)
  Using cached langchain-1.4.6-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_openai-1.7.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached streamlit-1.66.0-py3-none-any.whl.metadata (10 kB)
  Using cached llama_index_core-0.14.25-py3-none-any.whl.metadata (2.6 kB)
  Using cached llama_index_embeddings_openai-0.6.0-py3-none-any.whl.metadata (401 bytes)
  Using cached llama_index_llms_openai-0.7.10-py3-none-any.whl.metadata (3.0 kB)
  Using cached nltk-3.10.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp-3.14.5-cp314-cp314-win_amd64.whl.metadata (8.5 kB)
  Using cached banks-2.5.1-py3-none-any.whl.metadata (12 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx-0.28.1-py3-none-any.


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    VectorStoreIndex
)

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

print("Libraries imported successfully!")

d:\2026\GENERATIVE AI BATCH 2026\langchain assignment rohan batch 4 2026\Document_QA_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


In [3]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "API key missing. Add OPENAI_API_KEY to your .env file."
    )

DOC_DIR = Path("documents")
DOC_DIR.mkdir(exist_ok=True)

files = [
    p for p in DOC_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in {".pdf", ".txt"}
]

print("Document folder:", DOC_DIR.resolve())
print("Files found:", [p.name for p in files])

Document folder: D:\2026\GENERATIVE AI BATCH 2026\langchain assignment rohan batch 4 2026\Document_QA_RAG\documents
Files found: ['1810.04805v2.pdf', 'ReACT_2210.03629v3.pdf']


In [4]:
if not files:
    raise FileNotFoundError(
        "Add at least one PDF or TXT file to documents/."
    )

documents = SimpleDirectoryReader(
    input_dir=str(DOC_DIR),
    required_exts=[".pdf", ".txt"]
).load_data()

print("Loaded document pages or sections:", len(documents))

print("\nSample extracted text:")
print(documents[0].get_content()[:1000])

Loaded document pages or sections: 2

Sample extracted text:
%PDF-1.5
%
134 0 obj
<< /Filter /FlateDecode /Length 3888 >>
stream
 F۽Y޸7?|߷߼^7;KĻM3O7os&<mkoXEc&MO:/6ʹ)L'zLkٰʗf26]m̴̬Ҽȋ+Ӭ촻;i|Y4Di+NE:rLצF'"hehMLoaded document pages or sections: 2

Sample extracted text:
%PDF-1.5
%
134 0 obj
<< /Filter /FlateDecode /Length 3888 >>
stream
et?SEPVSMvtdY>&7oAMZ3 (ȗ(Ȍt]2Y^
XW|\	c+h&Yk+ilӭʹ2QCcoDߨL;@>Їs~īNbG:ݲa'$HٖU2b1Ǥ`#\)EIrB,:zF|
C"ǧbvDEf/>k/7!/:J^;U½l"}|G-#2/<{}a$#_C}At<j
a-u:j+1lt{W1BYף+uQ	ihH.P.	)>X I'L
B9ތq&77KV35#K$g~TltV:!WV_eL{JsZ^@]̩ ؔՓjS:4xdPL. L:&^,hNZ8Y\byShM9C2YzݤU;5<m`'iㆇGƽ{vAYO1yƏtW\"Rbc1;@K-	{D.	>\__-:V1D7A/x(2isFx},Ivjm˼Y٢ܼiZrM"7
Ӗ0Z	F	s400hͫܐLOWp|e
xtdMY[
{utJazl!Bjqz8!&h0,>u5A6G5)I1v`΂{qy,~U0Fm
7Շ# QpY]M4Ujsc]x!t221@3]fifXz>`n3a'wrB#V{Ivn octe9Z{QN(:oDGu'}xwLz-Z&$OW


In [5]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready!")

d:\2026\GENERATIVE AI BATCH 2026\langchain assignment rohan batch 4 2026\Document_QA_RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rohan\AppData\Local\llama_index\llama_index\Cache\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\2026\GENERA

Embedding model ready!


In [6]:
index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True
)

print("LlamaIndex vector index created!")

Generating embeddings: 100%|██████████| 933/933 [01:00<00:00, 15.38it/s]


LlamaIndex vector index created!


In [7]:
retriever = index.as_retriever(
    similarity_top_k=4
)

print("Retriever created!")

Retriever created!


In [8]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("Language model initialized!")

Language model initialized!


In [9]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an AI document question-answering assistant.

Use only the supplied context to answer the question.

Rules:
- Do not invent facts.
- If the answer is not present, say:
  "I could not find that information in the uploaded documents."
- Be clear and concise.
- Refer to source labels when useful."""
    ),
    (
        "human",
        """Context:
{context}

Question:
{question}

Answer:"""
    )
])

print("Prompt template created!")

Prompt template created!


In [10]:
answer_chain = prompt | llm | StrOutputParser()

print("LangChain processing chain created!")

LangChain processing chain created!


In [11]:
def answer_question(question, top_k=4):
    question = question.strip()

    if not question:
        return {
            "answer": "Please enter a question.",
            "sources": []
        }

    # Retrieve relevant passages with LlamaIndex
    retriever = index.as_retriever(
        similarity_top_k=top_k
    )
    nodes = retriever.retrieve(question)

    if not nodes:
        return {
            "answer": (
                "I could not find relevant information "
                "in the uploaded documents."
            ),
            "sources": []
        }

    context_parts = []
    sources = []

    # Prepare context and source information
    for i, item in enumerate(nodes, start=1):
        text = item.node.get_content()
        metadata = item.node.metadata or {}

        filename = (
            metadata.get("file_name")
            or metadata.get("filename")
            or "Uploaded document"
        )

        page = (
            metadata.get("page_label")
            or metadata.get("page")
        )

        label = f"[Source {i}] {filename}"

        if page is not None:
            label += f", page {page}"

        context_parts.append(f"{label}\n{text}")

        sources.append({
            "label": label,
            "score": (
                round(float(item.score), 4)
                if item.score is not None
                else None
            ),
            "text": text
        })

    context = "\n\n".join(context_parts)

    # Generate the answer through LangChain
    answer = answer_chain.invoke({
        "context": context,
        "question": question
    })

    return {
        "answer": answer,
        "sources": sources
    }

print("RAG question-answering function ready!")

RAG question-answering function ready!


In [ ]:
question = input("Enter your question:what is reactive programming?")

result = answer_question(question)

print("\nANSWER")
print("=" * 50)
print(result["answer"])

print("\nSOURCE PASSAGES")
print("=" * 50)

for source in result["sources"]:
    print("\n", source["label"])
    print("Similarity score:", source["score"])
    print(source["text"][:1000])

In [ ]:
def summarize_documents(top_k=8):
    nodes = index.as_retriever(
        similarity_top_k=top_k
    ).retrieve("main topics key findings conclusions")

    if not nodes:
        return "No relevant passages found."

    context = "\n\n".join(
        item.node.get_content()
        for item in nodes
    )

    summary_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """Summarize only the supplied passages.
Use headings and bullet points.
Do not add unsupported facts.
Mention that the summary covers retrieved passages."""
        ),
        (
            "human",
            "Summarize the following content:\n{context}"
        )
    ])

    summary_chain = summary_prompt | llm | StrOutputParser()

    return summary_chain.invoke({"context": context})


print(summarize_documents())